# Week 83, Day 5: Data cleaning project

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w083d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('sales_raw.csv', 'w', encoding='utf-8') as f:
    f.write(' Order ID,Date ,Customer Name,Town,Item,Qty,Price,Paid?\nA001,2026-03-01,ada lovelace, York,Tomato plant,2,£4.50,yes\nA002,01/03/2026,Bo  Chen,leeds ,Basil,1,2.5,Y\nA003,2 Mar 2026,CY OKAFOR,YORK,tomato plants,3,£4.50,no\nA004,03/03/2026,Dee Patel,Hull,Compost (40L),1,"£1,200.00",TRUE\nA005,,Eli Novak,Bath,Basil,2,£2.50,n\nA006,31/02/2026,Fay Morgan,bath,Lavender,1,£6.00,yes\nA007,2026-03-05,Gus Adeyemi,Hull,Lavender,-1,£6.00,yes\nA008,2026-03-05,Hal Brooks,Leeds,Compost (40L),1,free,maybe\nA002,01/03/2026,Bo  Chen,leeds ,Basil,1,2.5,Y\nA009,2026-03-06,Ivy Shaw,York,Basil,,£2.50,yes\nA010,6 Mar 2026,ada lovelace,york,Lavender,2,£6,no\nA003,2026-03-02,Cy Okafor,York,Tomato plant,3,£4.50,no\nA011,07/03/2026,Jo Reyes,Leeds,Basil,4,£2.50,y\nA012,2026-03-07,Bo Chen,Leeds,Tomato plant,1,£4.50,yes\n')

**Predict before you run:** what will this print?



In [ ]:
import pandas as pd

raw = pd.read_csv("sales_raw.csv", dtype=str)
print(raw.columns.tolist()[:3])
print(raw["Town"].unique().tolist())
print(raw["Price"].tolist()[:4])
print(len(raw), raw.duplicated().sum(), raw[" Order ID"].duplicated().sum())

In [ ]:
import pandas as pd

raw = pd.read_csv("sales_raw.csv", dtype=str)
for column in raw.columns:
    values = raw[column]
    print(f"{column!r:16} missing={values.isna().sum()} distinct={values.nunique()}")
print(raw["Paid?"].value_counts().to_dict())

In [ ]:
import re
import pandas as pd

raw = pd.read_csv("sales_raw.csv", dtype=str)
names = [re.sub(r"[^a-z0-9]+", "_", c.strip().lower()).strip("_") for c in raw.columns]
df = raw.set_axis(names, axis=1)
print(df.columns.tolist())

towns = df["town"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
print(towns.value_counts().to_dict())
items = df["item"].str.strip().str.title().replace({"Tomato Plants": "Tomato Plant"})
print(sorted(items.unique()))

In [ ]:
import pandas as pd

dates = pd.Series(["2026-03-01", "01/03/2026", "2 Mar 2026", None, "31/02/2026"])
result = None
for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%d %b %Y"]:
    parsed = pd.to_datetime(dates, format=fmt, errors="coerce")
    result = parsed if result is None else result.fillna(parsed)
print(result.dt.strftime("%Y-%m-%d").tolist())

prices = pd.Series(["£4.50", "2.5", "£1,200.00", "free", None, "a lot"])
text = prices.str.strip().str.lower().str.replace("£", "").str.replace(",", "")
print(pd.to_numeric(text.replace("free", "0"), errors="coerce").tolist())

answers = pd.Series(["yes", "Y", "no", "TRUE", "maybe"])
print(answers.str.strip().str.lower().map({"yes": True, "y": True, "true": True, "no": False, "n": False}).tolist())

In [ ]:
import pandas as pd

df = pd.DataFrame({"order_id": ["A1", "A2", "A2", "A3", "A1"], "qty": [2, 1, 1, -1, 5]})
print(df.duplicated().sum(), df.duplicated(subset="order_id").sum())
step1 = df.drop_duplicates()
step2 = step1.drop_duplicates(subset="order_id", keep="first")
print(step2["order_id"].tolist())
rules = {"bad quantity": step2["qty"] <= 0}
for reason, failed in rules.items():
    print(reason, step2.loc[failed, "order_id"].tolist())

> **Key idea.** Read raw files with `dtype=str`, never edit them, and profile every column (missing values, distinct values) before cleaning. Standardise column names (strip, lower-case, non-alphanumerics to `_`), then text values (`strip`, collapse spaces, consistent case) and known variants with an explicit `replace` dict. Parse dates by trying each known format with `errors="coerce"` and filling gaps; parse money by removing symbols and separators, translating words, then `to_numeric(errors="coerce")`; map every accepted yes/no spelling and leave the rest missing. Remove exact duplicates, then key duplicates with a documented `keep` choice, after tidying text. Validate with named rule masks, keep the rows that pass, and report the rest with reasons. Count everything that changes.

### Exercise: A first look

Write `tidy_columns(df)`: a new DataFrame with each column name stripped, lower-cased, every run of characters other than letters and digits replaced by `_`, and `_` removed from the ends. Then `profile(df)`: a dict from each column name to `{"missing": n, "distinct": n}` (distinct counts ignore missing values).

In [ ]:
import re
import pandas as pd

def tidy_columns(df):
    ___

def profile(df):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_profile():\n    \"names you can type\"\n    raw = pd.read_csv(\"sales_raw.csv\", dtype=str)\n    df = tidy_columns(raw)\n    assert df.columns.tolist() == [\"order_id\", \"date\", \"customer_name\", \"town\", \"item\", \"qty\", \"price\", \"paid\"]\n    assert raw.columns.tolist()[0] == \" Order ID\", \"leave raw unchanged\"\n    p = profile(df)\n    assert p[\"date\"] == {\"missing\": 1, \"distinct\": 11} and p[\"qty\"] == {\"missing\": 1, \"distinct\": 5}\n    assert p[\"town\"] == {\"missing\": 0, \"distinct\": 9} and p[\"order_id\"][\"distinct\"] == 12\n    assert tidy_columns(pd.DataFrame(columns=[\"Total (\u00a3)\", \"  e-mail  \"])).columns.tolist() == [\"total\", \"e_mail\"]")

### Exercise: Dates in three formats

Write `parse_dates(series, formats=("%Y-%m-%d", "%d/%m/%Y", "%d %b %Y"))`: strip each value, try the formats in order, each value taking the first format that reads it, and return a Series of dates with `NaT` where none did. Then `date_report(series)`: a dict with `"parsed"` (how many values became dates), `"failed"` (the original non-missing values that couldn't be read, in order) and `"blank"` (how many were missing to begin with).

In [ ]:
import pandas as pd

FORMATS = ("%Y-%m-%d", "%d/%m/%Y", "%d %b %Y")

def parse_dates(series, formats=FORMATS):
    ___

def date_report(series):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_dates():\n    \"each format in turn\"\n    raw = pd.read_csv(\"sales_raw.csv\", dtype=str)\n    dates = parse_dates(raw[\"Date \"])\n    assert dates.dt.strftime(\"%Y-%m-%d\").fillna(\"-\").tolist()[:6] == [\"2026-03-01\", \"2026-03-01\", \"2026-03-02\", \"2026-03-03\", \"-\", \"-\"]\n    assert int(dates.isna().sum()) == 2 and dates.iloc[10].strftime(\"%d %B\") == \"06 March\"\n    assert date_report(raw[\"Date \"]) == {\"parsed\": 12, \"failed\": [\"31/02/2026\"], \"blank\": 1}\n    assert parse_dates(pd.Series([\" 2026-04-09 \"])).iloc[0].day == 9, \"strip first\"")

### Exercise: Prices and quantities

Write `parse_money(series)`: strip and lower-case, remove `£` and `,`, treat `"free"` as `0`, and return floats with `NaN` for anything unreadable or missing. Then `parse_quantity(series)`: whole numbers from text, as floats with `NaN` for missing or unreadable values (a column with gaps can't be integers).

In [ ]:
import pandas as pd

def parse_money(series):
    ___

def parse_quantity(series):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math\nimport pandas as pd\n\ndef test_money():\n    \"symbols, separators and words\"\n    raw = pd.read_csv(\"sales_raw.csv\", dtype=str)\n    prices = parse_money(raw[\"Price\"])\n    assert prices.tolist() == [4.5, 2.5, 4.5, 1200.0, 2.5, 6.0, 6.0, 0.0, 2.5, 2.5, 6.0, 4.5, 2.5, 4.5]\n    odd = parse_money(pd.Series([\"\u00a3 3.20\", \"lots\", None, \"FREE\"]))\n    assert odd.iloc[0] == 3.2 and math.isnan(odd.iloc[1]) and math.isnan(odd.iloc[2]) and odd.iloc[3] == 0\n    qty = parse_quantity(raw[\"Qty\"])\n    assert qty.tolist()[:8] == [2.0, 1.0, 3.0, 1.0, 2.0, 1.0, -1.0, 1.0] and math.isnan(qty.iloc[9])\n    assert math.isnan(parse_quantity(pd.Series([\"two\"])).iloc[0])")

### Exercise: One spelling for each thing

Write `tidy_text(series, fixes=None)`: strip, collapse inner runs of spaces to one, title case, then apply the `fixes` dict (if given) with `replace`. Then `variants(series)`: a dict from each tidied value to the sorted list of **different raw spellings** that became it, only for values with more than one spelling.

In [ ]:
import pandas as pd

def tidy_text(series, fixes=None):
    ___

def variants(series):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_text():\n    \"towns, names and items\"\n    raw = pd.read_csv(\"sales_raw.csv\", dtype=str)\n    assert tidy_text(raw[\"Town\"]).value_counts().to_dict() == {\"York\": 5, \"Leeds\": 5, \"Hull\": 2, \"Bath\": 2}\n    assert tidy_text(raw[\"Customer Name\"]).tolist()[:3] == [\"Ada Lovelace\", \"Bo Chen\", \"Cy Okafor\"]\n    items = tidy_text(raw[\"Item\"], {\"Tomato Plants\": \"Tomato Plant\"})\n    assert sorted(items.unique()) == [\"Basil\", \"Compost (40L)\", \"Lavender\", \"Tomato Plant\"]\n    v = variants(raw[\"Town\"])\n    assert v == {\"Bath\": [\"Bath\", \"bath\"], \"Leeds\": [\"Leeds\", \"leeds \"], \"York\": [\" York\", \"YORK\", \"York\", \"york\"]}")

### Exercise: Yes, no and maybe

Write `parse_bool(series)`: map `"yes"`, `"y"`, `"true"` and `"1"` to `True` and `"no"`, `"n"`, `"false"` and `"0"` to `False`, ignoring case and spaces at the ends; anything else becomes missing. Then `unknown_answers(series)`: the sorted distinct raw values that didn't map (ignoring missing ones).

In [ ]:
import pandas as pd

def parse_bool(series):
    ___

def unknown_answers(series):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_bool():\n    \"every accepted spelling\"\n    raw = pd.read_csv(\"sales_raw.csv\", dtype=str)\n    paid = parse_bool(raw[\"Paid?\"])\n    assert paid.isna().tolist() == [False] * 7 + [True] + [False] * 6\n    assert paid.dropna().tolist() == [True, True, False, True, False, True, True, True, True, False, False, True, True]\n    assert unknown_answers(pd.Series([\"yes\", \"maybe\", \" Perhaps\", None, \"maybe\", \"N\"])) == [\" Perhaps\", \"maybe\"]")

### Exercise: Duplicates and rules

Write `remove_duplicates(df)`: drop exact duplicate rows, then rows whose `order_id` was already seen (keeping the first), returning `(new_df, exact, by_id)` with the two counts. Then `validate(df)` for a DataFrame whose `date` column holds dates and `qty` and `price` floats: the rules are `"no date"` (missing date), `"bad quantity"` (missing or not above 0), `"bad price"` (missing) and `"price looks wrong"` (above 500). Return `(good, problems)`: the rows that break no rule (index renumbered) and a list of `(order_id, reason)` for every broken rule, sorted.

In [ ]:
import pandas as pd

def remove_duplicates(df):
    ___

def validate(df):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\nimport pandas as pd\n\ndef test_duplicates():\n    \"exact first, then by key\"\n    df = pd.DataFrame({\"order_id\": [\"A1\", \"A2\", \"A2\", \"A3\", \"A1\"], \"qty\": [\"2\", \"1\", \"1\", \"4\", \"5\"]})\n    out, exact, by_id = remove_duplicates(df)\n    assert out[\"order_id\"].tolist() == [\"A1\", \"A2\", \"A3\"] and (exact, by_id) == (1, 1)\n\ndef test_validate():\n    \"named rules\"\n    df = pd.DataFrame({\"order_id\": [\"A1\", \"A2\", \"A3\", \"A4\", \"A5\"],\n                       \"date\": pd.to_datetime([\"2026-03-01\", None, \"2026-03-02\", \"2026-03-03\", \"2026-03-04\"]),\n                       \"qty\": [2.0, 1.0, -1.0, np.nan, 1.0],\n                       \"price\": [4.5, np.nan, 2.0, 2.5, 900.0]})\n    good, problems = validate(df)\n    assert good[\"order_id\"].tolist() == [\"A1\"] and good.index.tolist() == [0]\n    assert problems == [(\"A2\", \"bad price\"), (\"A2\", \"no date\"), (\"A3\", \"bad quantity\"), (\"A4\", \"bad quantity\"), (\"A5\", \"price looks wrong\")]")

### Bug hunt: York's missing sales

York had three sales, but the filter finds only one, because the town was typed in different ways: it prints `York sales: 1`. Tidy the town before filtering so it prints `York sales: 3`.

In [ ]:
import pandas as pd

towns = pd.Series(["York", " york", "Leeds", "YORK "])
print("York sales:", int((towns == "York").sum()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"strip and case\"\n    expect_output(\"York sales: 3\")")

### Bug hunt: A thousand pounds of compost

Removing the pound sign isn't enough: `"1,200.00"` still has a comma, and the conversion crashes with `ValueError`. Make it print `[4.5, 1200.0]`.

In [ ]:
import pandas as pd

prices = pd.Series(["£4.50", "£1,200.00"])
print(prices.str.replace("£", "").astype(float).tolist())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separators too\"\n    expect_output(\"[4.5, 1200.0]\")")

### Bug hunt: Duplicates that stayed

`drop_duplicates` returns a new DataFrame; it doesn't change the one it's called on. So the count doesn't move: `rows: 4`. Keep the result so it prints `rows: 3`.

In [ ]:
import pandas as pd

df = pd.DataFrame({"order_id": ["A1", "A2", "A2", "A3"], "qty": [2, 1, 1, 4]})
df.drop_duplicates()
print("rows:", len(df))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"methods return new tables\"\n    expect_output(\"rows: 3\")")

### Puzzle: The cleaning pipeline

Arrange the pipeline in a sensible order: read everything as text, fix names, tidy text **before** looking for duplicates, convert, then validate and write. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
import pandas as pd

s = pd.Series([" a", "A", "b ", None])
p = ___   # s.str.strip().str.lower().nunique()
q = ___   # int(s.isna().sum())
r = ___   # pd.to_numeric(pd.Series(["3", "x"]), errors="coerce").isna().tolist()
d = pd.DataFrame({"k": [1, 1, 2], "v": [5, 5, 6]})
t = ___   # int(d.duplicated().sum())
u = ___   # True or False: does d.drop_duplicates() change d itself?

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == 1 and r == [False, True], \"p, q and r\"\n    assert t == 1 and u is False, \"t and u\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the whole pipeline

Write `clean_sales(path, out_path, fixes)`: read the raw file as text and clean it with today's steps (the given helpers): tidy the column names; tidy `customer_name`, `town` and `item` (with `fixes` for items); remove duplicates; parse `date`, `qty`, `price` and `paid`; validate. Add a `total` column (`qty × price`, 2 places), sort by date then order id, and write the clean rows to `out_path` with dates as `YYYY-MM-DD` and no index. Return `(clean, report)`, where `report` has `rows_in`, `rows_out`, `exact_duplicates`, `id_duplicates`, `problems` (as `validate` gives them), `revenue` (2 places), `revenue_by_town` (a dict, 2 places), and `unpaid` (sorted order ids whose `paid` is `False` **or** unknown).

In [ ]:
import re
import pandas as pd

def tidy_columns(df):
    return df.set_axis([re.sub(r"[^a-z0-9]+", "_", c.strip().lower()).strip("_") for c in df.columns], axis=1)

def tidy_text(series, fixes=None):
    out = series.str.strip().str.replace(r"\s+", " ", regex=True).str.title()
    return out.replace(fixes) if fixes else out

def parse_dates(series, formats=("%Y-%m-%d", "%d/%m/%Y", "%d %b %Y")):
    text, result = series.str.strip(), None
    for fmt in formats:
        parsed = pd.to_datetime(text, format=fmt, errors="coerce")
        result = parsed if result is None else result.fillna(parsed)
    return result

def parse_money(series):
    text = series.str.strip().str.lower().str.replace("£", "", regex=False).str.replace(",", "", regex=False)
    return pd.to_numeric(text.str.strip().replace("free", "0"), errors="coerce").astype(float)

def parse_quantity(series):
    return pd.to_numeric(series.str.strip(), errors="coerce").astype(float)

ANSWERS = {"yes": True, "y": True, "true": True, "1": True, "no": False, "n": False, "false": False, "0": False}

def parse_bool(series):
    return series.str.strip().str.lower().map(ANSWERS)

def remove_duplicates(df):
    step1 = df.drop_duplicates()
    step2 = step1.drop_duplicates(subset="order_id", keep="first")
    return step2, len(df) - len(step1), len(step1) - len(step2)

def validate(df):
    rules = {"no date": df["date"].isna(), "bad quantity": df["qty"].isna() | (df["qty"] <= 0),
             "bad price": df["price"].isna(), "price looks wrong": df["price"] > 500}
    problems, broken = [], pd.Series(False, index=df.index)
    for reason, failed in rules.items():
        problems += [(oid, reason) for oid in df.loc[failed, "order_id"]]
        broken |= failed
    return df[~broken].reset_index(drop=True), sorted(problems)

def clean_sales(path, out_path, fixes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\nFIXES = {\"Tomato Plants\": \"Tomato Plant\"}\n\ndef test_clean():\n    \"the clean table\"\n    clean, report = clean_sales(\"sales_raw.csv\", \"sales_clean.csv\", FIXES)\n    assert clean[\"order_id\"].tolist() == [\"A001\", \"A002\", \"A003\", \"A008\", \"A010\", \"A011\", \"A012\"]\n    assert clean[\"total\"].tolist() == [9.0, 2.5, 13.5, 0.0, 12.0, 10.0, 4.5]\n    assert clean[\"customer_name\"].tolist()[:3] == [\"Ada Lovelace\", \"Bo Chen\", \"Cy Okafor\"]\n    assert clean[\"item\"].tolist()[2] == \"Tomato Plant\"\n\ndef test_report():\n    \"everything counted\"\n    clean, report = clean_sales(\"sales_raw.csv\", \"sales_clean.csv\", FIXES)\n    assert report[\"rows_in\"] == 14 and report[\"rows_out\"] == 7\n    assert (report[\"exact_duplicates\"], report[\"id_duplicates\"]) == (1, 1)\n    assert report[\"problems\"] == [(\"A004\", \"price looks wrong\"), (\"A005\", \"no date\"), (\"A006\", \"no date\"),\n                                  (\"A007\", \"bad quantity\"), (\"A009\", \"bad quantity\")]\n    assert report[\"revenue\"] == 51.5 and report[\"revenue_by_town\"] == {\"Leeds\": 17.0, \"York\": 34.5}\n    assert report[\"unpaid\"] == [\"A003\", \"A008\", \"A010\"]\n\ndef test_file():\n    \"a clean file on disk\"\n    clean_sales(\"sales_raw.csv\", \"sales_clean.csv\", FIXES)\n    saved = pd.read_csv(\"sales_clean.csv\", dtype=str)\n    assert saved.columns.tolist()[:3] == [\"order_id\", \"date\", \"customer_name\"] and len(saved) == 7\n    assert saved[\"date\"].tolist()[:3] == [\"2026-03-01\", \"2026-03-01\", \"2026-03-02\"]\n    assert \"Unnamed: 0\" not in saved.columns")

## Warm-up

Fourteen rows, and almost every column has a problem. Notice `dtype=str`: reading **every column as text** stops pandas guessing types from messy values (a column with `"free"` in it would become text anyway, and `"007"` would lose its zeros). You then convert each column deliberately, knowing what to do with the values that don't fit.

## Look before you clean

Before changing anything, **profile** the file: for each column, how many values are missing, how many distinct values there are, and what the distinct values look like (`unique()` or `value_counts()` on the short columns). That tells you which problems exist, and gives you the numbers to compare against afterwards.

Two rules make cleaning trustworthy:

- **Never edit the raw file.** Read it, clean a copy, write the result to a new file. If a rule turns out to be wrong, fix the code and run it again.
- **Count everything you change or drop.** A pipeline that silently loses rows is worse than none; the report at the end says how many rows came in, how many went out, and why each missing one went.

## Names and text

**Column names** first: strip spaces, lower-case, and replace anything that isn't a letter or digit with `_`, so `" Order ID"` becomes `order_id` and `"Paid?"` becomes `paid`. Then every later step can use `df["order_id"]`.

**Text values** next: `.str.strip()` removes spaces at the ends, `.str.replace(r"\s+", " ", regex=True)` collapses runs of spaces inside, and a consistent case (`.str.title()` for names and places) makes `" York"`, `"YORK"` and `"york"` the same value. Some variants are different **words** (`"tomato plants"` against `"Tomato plant"`): fix those with an explicit dict of known variants, `series.replace({"Tomato Plants": "Tomato Plant"})`. Keep that dict in one place; it's knowledge about the data that should be visible, not buried in code.

## Dates, money and answers

**Dates in several formats**: try each known format in turn with `pd.to_datetime(text, format=..., errors="coerce")`, and fill the gaps left by one format with the results of the next (`result.fillna(parsed)`). A value no format accepts (an empty cell, or 31 February) stays `NaT`, to be reported. Don't let pandas guess formats on mixed data: `01/03/2026` is the 1st of March here, but a guess could make it the 3rd of January.

**Money**: remove the symbol and the thousands separators with `.str.replace`, translate known words (`"free"` is 0), then `pd.to_numeric(text, errors="coerce")`, which turns anything still unreadable into `NaN` instead of crashing.

**Yes/no answers**: lower-case and strip, then `map` a dict of every accepted spelling to `True` or `False`. Anything else (`"maybe"`) maps to missing, which is honest: nobody knows.

## Duplicates and rules

**Exact duplicates** (a row entered twice) are found with `df.duplicated()` and removed with `df.drop_duplicates()`. **Key duplicates** are rows that share an id but differ elsewhere (here order `A003` was entered twice, once in each format). Which copy is right is a decision: keeping the **first** is a common, documented choice: `drop_duplicates(subset="order_id", keep="first")`. Do exact duplicates first, so the count of each is meaningful, and do both **after** tidying the text, so `"Bo  Chen"` and `"Bo Chen"` count as the same.

Then **validate**: write each rule as a named mask (no date; quantity missing or not positive; price missing; price implausibly high), collect the failing rows with their reasons, and keep the rows that pass every rule. Rejected rows aren't deleted from the world: they go into the report, so someone can fix them at the source.

## Practice

## Quiz

## Daily challenge

## Before you go

Run your pipeline, then open `sales_clean.csv` and the report side by side and check one row of each kind by hand: a row that was tidied, a duplicate that was removed, and a row that was rejected. Then imagine next fortnight's file has a fourth date format and a new spelling of "Leeds": which single line of your code changes for each? Tomorrow is the Week 83 **Forge**: a whole analysis, from a raw file to answers and charts.